# 2025년 서울 지하철 이용량 시계열 분석

**원본:** 서울 열린데이터광장 OA-12921  
**범위:** 서울교통공사 관할 1~8호선, 2025년

이 노트북은 공식 2025 원본의 역별 승차·하차와 20개 시간대 값을 날짜별로 합친 `data/processed/seoul_metro_daily_2025.csv`를 분석합니다.
처리 데이터가 없다면 먼저 다음을 실행합니다.

```bash
python scripts/prepare_data.py
```

분석 질문:
1. 연중 이용량의 장·단기 추세는 어떻게 나타나는가?
2. 요일별 이용량 차이는 얼마나 큰가?
3. 전일 대비 급격한 변화가 나타난 날짜는 언제인가?
4. 월별 평균 이용량에는 어떤 차이가 있는가?


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data/processed/seoul_metro_daily_2025.csv")
if not DATA.exists():
    raise FileNotFoundError(
        "처리 데이터가 없습니다. 터미널에서 `python scripts/prepare_data.py`를 먼저 실행하세요."
    )

df = pd.read_csv(DATA, parse_dates=["date"])
df.head()


,date,passengers,weekday_num,weekday,ma7,ma30,change_rate_pct,month
0,2025-01-01,3949907,2,Wed,NaN,NaN,NaN,1
1,2025-01-02,9300390,3,Thu,NaN,NaN,135.458455,1
2,2025-01-03,9798970,4,Fri,NaN,NaN,5.360850,1
3,2025-01-04,6958991,5,Sat,NaN,NaN,-28.982424,1
4,2025-01-05,4866114,6,Sun,NaN,NaN,-30.074432,1


## 1. 데이터 기본 정보 및 품질 확인

관찰과 해석을 분리하기 전에 기간, 데이터 포인트, 결측치, 값 범위를 확인합니다.
공식 파일의 완전 빈 행 134개만 전처리에서 제외했습니다. 데이터 행의 결측·숫자 오류·음수는 실패로 처리합니다.
이동평균과 변화율 첫 구간의 결측은 계산 창이 아직 채워지지 않아 생기는 정상적인 파생 결측입니다.
공휴일·연휴처럼 실제 수요 변화로 인해 낮아진 값은 단순 이상치로 삭제하지 않습니다.


In [2]:
print("기간:", df["date"].min().date(), "~", df["date"].max().date())
print("데이터 포인트:", len(df))
print("\n결측치")
print(df.isna().sum())
print("\npassengers 요약")
display(df["passengers"].describe())

assert pd.DatetimeIndex(df["date"]).equals(pd.date_range("2025-01-01", "2025-12-31"))
assert not df["date"].duplicated().any()
assert df["passengers"].notna().all()
assert (df["passengers"] > 0).all()


기간: 2025-01-01 ~ 2025-12-31
데이터 포인트: 365

결측치
date                0
passengers          0
weekday_num         0
weekday             0
ma7                 6
ma30               29
change_rate_pct     1
month               0
dtype: int64

passengers 요약


count    3.650000e+02
mean     8.881270e+06
std      1.977776e+06
min      2.907370e+06
25%      7.332468e+06
50%      9.831144e+06
75%      1.035403e+07
max      1.134948e+07
Name: passengers, dtype: float64

## 2. 이동평균

- 7일 이동평균: 주간 반복 변동 완화
- 30일 이동평균: 월 수준의 기저 추세 확인


In [3]:
df["ma7"] = df["passengers"].rolling(7).mean()
df["ma30"] = df["passengers"].rolling(30).mean()

plt.figure(figsize=(12,5))
plt.plot(df["date"], df["passengers"]/1e6, alpha=.4, label="Daily")
plt.plot(df["date"], df["ma7"]/1e6, label="7-day MA")
plt.plot(df["date"], df["ma30"]/1e6, linewidth=2, label="30-day MA")
plt.title("Seoul Metro Daily Ridership, 2025")
plt.ylabel("Passengers (millions)")
plt.legend()
plt.show()


C:\Users\proje\AppData\Local\Temp\ipykernel_22228\256173935.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 요일별 패턴

요일별 평균을 비교해 반복적인 주간 계절성을 확인합니다.


In [4]:
weekday_order = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
weekday_mean = df.groupby("weekday")["passengers"].mean().reindex(weekday_order)
display(weekday_mean)

weekday_mean.plot(kind="bar", figsize=(8,5), title="Average Ridership by Day of Week, 2025")
plt.ylabel("Passengers")
plt.show()


weekday
Mon    9.453614e+06
Tue    9.672598e+06
Wed    9.836882e+06
Thu    9.882744e+06
Fri    1.018995e+07
Sat    7.549139e+06
Sun    5.565590e+06
Name: passengers, dtype: float64

C:\Users\proje\AppData\Local\Temp\ipykernel_22228\2251402589.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. 전일 대비 변화율

급격한 변화 날짜를 찾되, 데이터만으로 원인을 단정하지 않습니다.


In [5]:
df["change_rate_pct"] = df["passengers"].pct_change() * 100

plt.figure(figsize=(12,5))
plt.plot(df["date"], df["change_rate_pct"])
plt.axhline(0, linewidth=.8)
plt.title("Day-over-Day Ridership Change, 2025")
plt.ylabel("Change (%)")
plt.show()

top_changes = (
    df.dropna(subset=["change_rate_pct"])
      .assign(abs_change=lambda x: x["change_rate_pct"].abs())
      .nlargest(10, "abs_change")
)
display(top_changes[["date","passengers","change_rate_pct"]])


C:\Users\proje\AppData\Local\Temp\ipykernel_22228\1739322689.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,date,passengers,change_rate_pct
1,2025-01-02,9300390,135.458455
30,2025-01-31,8144419,112.226499
33,2025-02-03,9430732,98.326949
5,2025-01-06,9478276,94.781216
271,2025-09-29,10166966,90.508639
159,2025-06-09,9865001,88.586999
285,2025-10-13,9642410,86.693910
12,2025-01-13,9465476,86.106113
229,2025-08-18,9304241,85.812976
40,2025-02-10,9498079,85.510370


## 5. 월별 평균

월별 평균을 비교하되, 계절 효과와 공휴일 수/평일 수 효과를 구분해야 합니다.


In [6]:
monthly = df.groupby("month")["passengers"].mean()
display(monthly)

monthly.plot(kind="bar", figsize=(9,5), title="Average Daily Ridership by Month, 2025")
plt.ylabel("Passengers")
plt.show()


month
1     7.712039e+06
2     8.797946e+06
3     9.068680e+06
4     9.460110e+06
5     9.028761e+06
6     8.769121e+06
7     8.887446e+06
8     8.368334e+06
9     9.355776e+06
10    8.406913e+06
11    9.315663e+06
12    9.440764e+06
Name: passengers, dtype: float64

C:\Users\proje\AppData\Local\Temp\ipykernel_22228\811704249.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. 핵심 통계

아래 결과는 REPORT.md의 관찰(Fact) 근거로 사용합니다.


In [7]:
weekday_avg = df[df["weekday_num"] < 5]["passengers"].mean()
weekend_avg = df[df["weekday_num"] >= 5]["passengers"].mean()
peak = df.loc[df["passengers"].idxmax()]
low = df.loc[df["passengers"].idxmin()]

print(f"일평균: {df['passengers'].mean():,.0f}명")
print(f"중앙값: {df['passengers'].median():,.0f}명")
print(f"표준편차(365일 모집단): {df['passengers'].std(ddof=0):,.0f}명")
print(f"평일 평균: {weekday_avg:,.0f}명")
print(f"주말 평균: {weekend_avg:,.0f}명")
print(f"평일이 주말보다: {(weekday_avg/weekend_avg-1)*100:.1f}% 높음")
print(f"최대: {peak['date'].date()} / {int(peak['passengers']):,}명")
print(f"최소: {low['date'].date()} / {int(low['passengers']):,}명")


일평균: 8,881,270명
중앙값: 9,831,144명
표준편차(365일 모집단): 1,975,065명
평일 평균: 9,807,271명
주말 평균: 6,557,365명
평일이 주말보다: 49.6% 높음
최대: 2025-12-19 / 11,349,481명
최소: 2025-01-29 / 2,907,370명


## 7. 해석 원칙

- **관찰(Fact):** 평일 일평균 9,807,271명과 주말 일평균 6,557,365명, 4월 최고 월별 일평균 9,460,110명, 1월 최저 월별 일평균 7,712,039명은 위 계산으로 확인됩니다.
- **해석(가설):** 주중 반복 이동이나 달력 효과가 차이에 영향을 줄 수 있지만, 이 집계만으로 원인을 확정할 수 없습니다.
- **추가 검증:** 공휴일, 날씨, 행사, 운영 이슈 등의 외부 데이터로 가설을 확인

예를 들어 특정 날짜의 이용량이 크게 감소했더라도,
해당 날짜가 공휴일이었다는 외부 근거를 확인하기 전에는
“공휴일 때문에 감소했다”고 인과적으로 단정하지 않습니다.
